# Ablation V4 — Full PGA + Binary Prompt
PSG + CAD đầy đủ nhưng train với prompt nhị phân (không GaussianBlur).

In [ ]:
import os, sys, subprocess
import gdown

BASE = '/content'
os.chdir(BASE)

REPO = 'https://github.com/ThongLuc2k3/Prompt-Guided-XRay-Segmentation.git'
if not os.path.exists(f'{BASE}/pga-repo'):
    !git clone -q -b TN_B_ON {REPO} {BASE}/pga-repo
sys.path.insert(0, f'{BASE}/pga-repo')

DS_PATH = f'{BASE}/pga-repo/dataset_BTXRD'
if not os.path.exists(DS_PATH):
    gdown.download('https://drive.google.com/uc?id=1fU7KPln7joaa3EZZtGn-VKeg9i4AmPG3',
                   f'{BASE}/dataset_BTXRD.zip', quiet=False)
    !unzip -oq {BASE}/dataset_BTXRD.zip -d {BASE}/pga-repo/

os.makedirs('checkpoints', exist_ok=True)
!pip install -q tqdm opencv-python timm scipy
print("✅ Setup done")

# ── Mount Google Drive để lưu checkpoint ────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    import os; os.makedirs('/content/drive/MyDrive/ablation_checkpoints', exist_ok=True)
    print('✅ Drive mounted')
except Exception as _e:
    print(f'⚠️ Drive not mounted ({_e}) — lưu local only')


In [ ]:
USE_ENCODER_PROMPT = True   # PSG + CAD
BINARY_PROMPT      = True   # binary bbox mask (no GaussianBlur)
VARIANT_NAME       = 'V4 — Full PGA + Binary prompt'


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, cv2, json as _json, glob, os
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from scipy.ndimage import binary_erosion, distance_transform_edt

# ── clear module cache then import model ──────────────────────────────────
import sys
sys.path.insert(0, '/content/pga-repo')
for _k in list(sys.modules.keys()):
    if 'models' in _k or 'prompt_unet' in _k: del sys.modules[_k]
from models.networks.prompt_unet_2D import PGA_UNet

IMG_SIZE = 512
ZOOM_R   = 0.30
BATCH    = 4
EPOCHS   = 100
LR       = 1e-4
PATIENCE = 15
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class BTXRDDataset(Dataset):
    def __init__(self, split='train'):
        self.samples = []
        base = '/content/pga-repo/dataset_BTXRD'
        img_dir = f'{base}/{split}/images'
        ann_dir = f'{base}/{split}/annotations'
        for jf in sorted(glob.glob(f'{ann_dir}/*.json')):
            stem = os.path.splitext(os.path.basename(jf))[0]
            img_path = next((f'{img_dir}/{stem}{e}' for e in ('.png','.jpg','.jpeg')
                             if os.path.exists(f'{img_dir}/{stem}{e}')), None)
            if not img_path: continue
            data = _json.load(open(jf, encoding='utf-8'))
            for shp in [s for s in data.get('shapes',[]) if s.get('shape_type')=='polygon']:
                pts = np.array(shp['points'], dtype=np.float32)
                self.samples.append((img_path, pts))
    def __len__(self): return len(self.samples)
    def _make_prompt(self, b):
        x1,y1,x2,y2 = int(b[0]),int(b[1]),int(b[2]),int(b[3])
        hm = np.zeros((IMG_SIZE,IMG_SIZE), dtype=np.float32)
        if x2>x1 and y2>y1:
            hm[y1:y2,x1:x2] = 1.0
            if not BINARY_PROMPT:
                hm = cv2.GaussianBlur(hm,(31,31),0)
        return hm
    def __getitem__(self, idx):
        img_path, pts = self.samples[idx]
        img_bgr = cv2.imread(img_path); H0,W0 = img_bgr.shape[:2]
        sx,sy = IMG_SIZE/W0, IMG_SIZE/H0
        pts_s = pts*np.array([sx,sy])
        img512 = cv2.resize(img_bgr,(IMG_SIZE,IMG_SIZE))
        gray = cv2.cvtColor(img512, cv2.COLOR_BGR2GRAY)
        gt = np.zeros((IMG_SIZE,IMG_SIZE), dtype=np.uint8)
        cv2.fillPoly(gt,[pts_s.astype(np.int32)],255)
        xs,ys = pts_s[:,0],pts_s[:,1]
        bw,bh = xs.max()-xs.min(), ys.max()-ys.min()
        b = [max(0,xs.min()-bw*ZOOM_R), max(0,ys.min()-bh*ZOOM_R),
             min(IMG_SIZE,xs.max()+bw*ZOOM_R), min(IMG_SIZE,ys.max()+bh*ZOOM_R)]
        hm = self._make_prompt(b)
        img_n = (gray.astype(np.float32)/255.-0.5)/0.5
        return (torch.from_numpy(img_n).unsqueeze(0).float(),
                torch.from_numpy(hm).unsqueeze(0).float(),
                torch.from_numpy(gt.astype(np.float32)/255.).unsqueeze(0).float())

def dice_loss(p, t, eps=1e-6):
    p = torch.sigmoid(p)
    i=(p*t).sum(dim=(2,3))
    return (1-(2*i+eps)/(p.sum(dim=(2,3))+t.sum(dim=(2,3))+eps)).mean()
def loss_fn(logit, t): return F.binary_cross_entropy_with_logits(logit,t)+dice_loss(logit,t)

def eval_dice(model, loader):
    model.eval(); dices=[]
    with torch.no_grad():
        for img,hm,gt in loader:
            img,hm,gt = img.to(DEVICE),hm.to(DEVICE),gt.to(DEVICE)
            p=(torch.sigmoid(model(img,hm))>0.5).float()
            i=(p*(gt>0.5)).sum(dim=(2,3)).float()
            u=p.sum(dim=(2,3))+(gt>0.5).sum(dim=(2,3)).float()
            dices+=((2*i+1e-6)/(u+1e-6)).cpu().numpy().flatten().tolist()
    return float(np.mean(dices))

# ── Build model & dataloaders ──────────────────────────────────────────────
model = PGA_UNet(in_channels=1, n_classes=1,
                 use_encoder_prompt=USE_ENCODER_PROMPT).to(DEVICE)
print(f'Model: {type(model).__name__}  params={sum(p.numel() for p in model.parameters()):,}')
print(f'BINARY_PROMPT={BINARY_PROMPT}  USE_ENCODER_PROMPT={USE_ENCODER_PROMPT}')

train_dl = DataLoader(BTXRDDataset('train'), BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(BTXRDDataset('val'),   BATCH, shuffle=False, num_workers=2)
test_dl  = DataLoader(BTXRDDataset('test'),  1,     shuffle=False, num_workers=2)

# ── Training loop ──────────────────────────────────────────────────────────
optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(optim, factor=0.5, patience=5)

best_dice, no_imp = 0.0, 0
for ep in range(1, EPOCHS+1):
    model.train(); total=0
    for img,hm,gt in tqdm(train_dl, desc=f'Ep{ep:03d}', leave=True):
        img,hm,gt = img.to(DEVICE),hm.to(DEVICE),gt.to(DEVICE)
        loss = loss_fn(model(img,hm), gt)
        optim.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optim.step(); total += loss.item()
    val_d = eval_dice(model, val_dl)
    sched.step(1-val_d)
    if val_d > best_dice:
        best_dice=val_d
        torch.save(model.state_dict(),'checkpoints/best.pth')
        try:
            import shutil as _sh
            _vname = VARIANT_NAME.replace(' ', '_').replace('—','').replace(',','').strip('_')
            _sh.copy('checkpoints/best.pth', f'/content/drive/MyDrive/ablation_checkpoints/{_vname}_best.pth')
        except: pass
        no_imp=0
    else:
        no_imp+=1
        if no_imp>=PATIENCE: print(f'  Early stop @ ep{ep}'); break
    print(f'  ep{ep:03d}  loss={total/len(train_dl):.4f}  val_dice={val_d:.4f}  best={best_dice:.4f}')
print(f'\n✅ Training done.  Best val Dice = {best_dice:.4f}')

# ── Evaluation on test set ─────────────────────────────────────────────────
model.load_state_dict(torch.load('checkpoints/best.pth', map_location=DEVICE))
model.eval()
KEYS=['dice','iou','pre','rec','hd95','cbl']; results=[]
with torch.no_grad():
    for img,hm,gt in tqdm(test_dl,'Eval'):
        img,hm,gt = img.to(DEVICE),hm.to(DEVICE),gt.to(DEVICE)
        prob = torch.sigmoid(model(img,hm))[0,0].cpu().numpy()
        pm=(prob>0.5).astype(np.float32); gm=gt[0,0].cpu().numpy()
        tp=(pm*gm).sum(); fp=(pm*(1-gm)).sum(); fn=((1-pm)*gm).sum(); eps=1e-6
        dice=float((2*tp+eps)/(2*tp+fp+fn+eps)); iou=float((tp+eps)/(tp+fp+fn+eps))
        pre=float((tp+eps)/(tp+fp+eps)); rec=float((tp+eps)/(tp+fn+eps))
        p,g=pm.astype(bool),gm.astype(bool); hd95=float(IMG_SIZE)
        if p.any() and g.any():
            pe=p^binary_erosion(p); ge=g^binary_erosion(g)
            d1=distance_transform_edt(~ge)[pe]; d2=distance_transform_edt(~pe)[ge]
            if len(d1) and len(d2): hd95=float(max(np.percentile(d1,95),np.percentile(d2,95)))
        cbl=0.
        if gm.sum()>0 and pm.sum()>0:
            ys,xs=np.where(gm>0.5); yp,xp=np.where(pm>0.5)
            d=np.sqrt((ys.max()-ys.min())**2+(xs.max()-xs.min())**2)+eps
            cbl=float(np.clip(1.-np.sqrt((xp.mean()-xs.mean())**2+(yp.mean()-ys.mean())**2)/d,0,1))
        results.append({'dice':dice,'iou':iou,'pre':pre,'rec':rec,'hd95':hd95,'cbl':cbl})
m={k:np.mean([r[k] for r in results]) for k in KEYS}
print(f"\n{'='*60}")
print(f"  TEST RESULTS ({VARIANT_NAME})")
print(f"  Dice={m['dice']:.4f}  IoU={m['iou']:.4f}  Pre={m['pre']:.4f}")
print(f"  Rec={m['rec']:.4f}   HD95={m['hd95']:.2f}px  CBL={m['cbl']:.4f}")
print(f"{'='*60}")


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

N_SHOW = 10
fig, axes = plt.subplots(N_SHOW, 5, figsize=(22, 4 * N_SHOW))
fig.suptitle(f'Visualization — {VARIANT_NAME}', fontsize=14, y=1.001)
col_titles = ['Ảnh gốc', 'Ảnh + Prompt', 'Dự đoán', 'Ground Truth', 'TP/FP/FN']
for ax, ct in zip(axes[0], col_titles):
    ax.set_title(ct, fontsize=11, fontweight='bold')

model.eval()
count = 0
with torch.no_grad():
    for img, hm, gt in test_dl:
        if count >= N_SHOW:
            break
        img_np  = (img[0, 0].cpu().numpy() * 0.5 + 0.5)
        hm_np   = hm[0, 0].cpu().numpy()
        gt_np   = (gt[0, 0].cpu().numpy() > 0.5).astype(float)
        prob_np = torch.sigmoid(model(img.to(DEVICE), hm.to(DEVICE)))[0, 0].cpu().numpy()
        pred_np = (prob_np > 0.5).astype(float)

        r   = results[count]
        row = axes[count]
        bg  = np.stack([img_np] * 3, axis=-1)

        # Col 0: Ảnh gốc
        row[0].imshow(img_np, cmap='gray', vmin=0, vmax=1)
        row[0].set_ylabel(f'#{count+1}', fontsize=9)

        # Col 1: Ảnh + Prompt
        row[1].imshow(img_np, cmap='gray', vmin=0, vmax=1)
        row[1].imshow(hm_np, cmap='hot', alpha=0.4, vmin=0, vmax=1)

        # Col 2: Dự đoán overlay (đỏ) + Dice/IoU/HD95
        pred_ov = bg.copy()
        pred_ov[..., 0] = np.clip(pred_ov[..., 0] + pred_np * 0.55, 0, 1)
        pred_ov[..., 1] = np.clip(pred_ov[..., 1] - pred_np * 0.2, 0, 1)
        pred_ov[..., 2] = np.clip(pred_ov[..., 2] - pred_np * 0.2, 0, 1)
        row[2].imshow(pred_ov)
        row[2].text(0.02, 0.98,
                    f"Dice={r['dice']:.3f}\nIoU={r['iou']:.3f}\nHD95={r['hd95']:.1f}px",
                    transform=row[2].transAxes, fontsize=7.5, color='yellow',
                    verticalalignment='top',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.65))

        # Col 3: Ground Truth overlay (xanh lá)
        gt_ov = bg.copy()
        gt_ov[..., 1] = np.clip(gt_ov[..., 1] + gt_np * 0.55, 0, 1)
        gt_ov[..., 0] = np.clip(gt_ov[..., 0] - gt_np * 0.2, 0, 1)
        gt_ov[..., 2] = np.clip(gt_ov[..., 2] - gt_np * 0.2, 0, 1)
        row[3].imshow(gt_ov)

        # Col 4: TP/FP/FN map + Pre/Rec/CBL
        tp = pred_np * gt_np
        fp = pred_np * (1 - gt_np)
        fn = (1 - pred_np) * gt_np
        inter = bg.copy() * 0.25
        inter[..., 1] = np.clip(inter[..., 1] + tp * 0.9, 0, 1)  # TP = green
        inter[..., 0] = np.clip(inter[..., 0] + fp * 1.0, 0, 1)  # FP = red
        inter[..., 2] = np.clip(inter[..., 2] + fn * 1.0, 0, 1)  # FN = blue
        row[4].imshow(inter)
        row[4].text(0.02, 0.98,
                    f"Pre={r['pre']:.3f}\nRec={r['rec']:.3f}\nCBL={r['cbl']:.3f}",
                    transform=row[4].transAxes, fontsize=7.5, color='yellow',
                    verticalalignment='top',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.65))

        for ax in row:
            ax.axis('off')
        count += 1

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='green',  label='TP (dú́ng)'),
    Patch(facecolor='red',    label='FP (dư ra)'),
    Patch(facecolor='blue',   label='FN (bỏ sót)'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, -0.01), frameon=True)

plt.tight_layout()
try:
    _vname = VARIANT_NAME.replace(' ', '_').replace('—','').replace(',','').strip('_')
    vis_path = f'/content/drive/MyDrive/ablation_checkpoints/{_vname}_vis10.png'
    plt.savefig(vis_path, dpi=100, bbox_inches='tight')
    print(f'✅ Saved → {vis_path}')
except Exception as _e:
    print(f'⚠️ Drive save failed: {_e}')
plt.show()
